# Intraday Liquidity Monitoring – Exploratory Analysis

This notebook demonstrates the end-to-end pipeline:
1. Load and explore transaction data
2. Aggregate into 5-minute buckets and compute liquidity position
3. Detect anomalous liquidity movements with Isolation Forest
4. Identify payment clusters responsible for liquidity events

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.data_processing import (
    load_transactions,
    aggregate_to_buckets,
    calculate_liquidity_position,
    aggregate_by_system,
    save_processed,
)
from src.feature_engineering import (
    build_bucket_features,
    build_transaction_features,
    select_anomaly_features,
    select_cluster_features,
    ANOMALY_FEATURE_COLS,
    CLUSTER_FEATURE_COLS,
)
from src.anomaly_detection import detect_anomalies, LiquidityAnomalyDetector
from src.clustering import (
    cluster_transactions,
    summarise_clusters,
    find_optimal_clusters,
    PaymentClusterer,
)
from src.visualization import (
    plot_liquidity_position,
    plot_anomalies,
    plot_clusters,
    plot_elbow,
)

print("All imports OK")

## 1. Load and Explore Transaction Data

In [ ]:
df = load_transactions("../data/raw/transactions.csv")
print(f"Transactions: {len(df):,}")
print(f"Date range : {df['timestamp'].min()} → {df['timestamp'].max()}")
df.head()

In [ ]:
print("Direction counts:")
print(df["direction"].value_counts())
print("
Payment system breakdown:")
print(df["payment_system"].value_counts())
print("
Counterparty type breakdown:")
print(df["counterparty_type"].value_counts())

In [ ]:
print("Amount statistics (M):")
(df["amount"] / 1e6).describe().round(3)

## 2. Aggregate to 5-Minute Buckets and Compute Liquidity Position

In [ ]:
agg = aggregate_to_buckets(df)
liq = calculate_liquidity_position(agg)
print(f"Buckets: {len(agg)}")
print(f"Liquidity position range: {liq['liquidity_position'].min()/1e6:,.1f}M – {liq['liquidity_position'].max()/1e6:,.1f}M")
liq[["bucket","total_inflow","total_outflow","net_flow","liquidity_position"]].head(10)

In [ ]:
fig_path = plot_liquidity_position(liq)
print(f"Saved: {fig_path}")
plt.figure(figsize=(14, 8))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
import matplotlib.dates as mdates
axes[0].plot(liq["bucket"], liq["liquidity_position"]/1e6, color="#1f77b4", linewidth=1.5)
axes[0].set_ylabel("Position (M)"); axes[0].set_title("Intraday Liquidity Position"); axes[0].grid(alpha=0.3)
colors = ["#2ca02c" if v >= 0 else "#d62728" for v in liq["net_flow"]]
axes[1].bar(liq["bucket"], liq["net_flow"]/1e6, color=colors, width=pd.Timedelta("4min"))
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Net flow (M)"); axes[1].set_xlabel("Time")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.tight_layout()
plt.show()

## 3. Per-System Flow Breakdown

In [ ]:
sys_agg = aggregate_by_system(df)
sys_pivot = sys_agg.pivot(index="bucket", columns="payment_system", values="net_flow").fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
sys_pivot.plot(ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Net Flow per Payment System (5-min buckets)")
ax.set_ylabel("Net flow (M)")
ax.set_xlabel("Time")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Anomaly Detection with Isolation Forest

In [ ]:
bucket_feats = build_bucket_features(liq)
bucket_with_anomalies = detect_anomalies(bucket_feats, ANOMALY_FEATURE_COLS, contamination=0.05)
n_anomalies = bucket_with_anomalies["is_anomaly"].sum()
print(f"Anomalous buckets detected: {n_anomalies} / {len(bucket_with_anomalies)}")
bucket_with_anomalies[bucket_with_anomalies["is_anomaly"]][["bucket","liquidity_position","net_flow","anomaly_score"]].head(10)

In [ ]:
fig_path = plot_anomalies(bucket_with_anomalies)
print(f"Saved: {fig_path}")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
normal = bucket_with_anomalies[~bucket_with_anomalies["is_anomaly"]]
anomalies = bucket_with_anomalies[bucket_with_anomalies["is_anomaly"]]
axes[0].plot(bucket_with_anomalies["bucket"], bucket_with_anomalies["liquidity_position"]/1e6, color="#1f77b4", linewidth=1.5, label="Liquidity position")
axes[0].scatter(anomalies["bucket"], anomalies["liquidity_position"]/1e6, color="#d62728", zorder=5, s=60, label=f"Anomaly ({n_anomalies} detected)")
axes[0].set_title("Anomaly Detection – Isolation Forest"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(bucket_with_anomalies["bucket"], bucket_with_anomalies["anomaly_score"], color="grey")
axes[1].scatter(anomalies["bucket"], anomalies["anomaly_score"], color="#d62728", zorder=5, s=40)
axes[1].set_ylabel("Anomaly score"); axes[1].set_xlabel("Time")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Payment Clustering with K-Means

In [ ]:
tx_feats = build_transaction_features(df)
cluster_feats = select_cluster_features(tx_feats)

# Find optimal k
scores = find_optimal_clusters(cluster_feats, k_range=range(2, 9))
fig_path = plot_elbow(scores)
print(f"Saved: {fig_path}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(scores["k"], scores["inertia"], marker="o"); ax1.set_title("Elbow Curve"); ax1.set_xlabel("k"); ax1.set_ylabel("Inertia"); ax1.grid(alpha=0.3)
ax2.plot(scores["k"], scores["silhouette"], marker="o", color="orange"); ax2.set_title("Silhouette Score"); ax2.set_xlabel("k"); ax2.set_ylabel("Score"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
N_CLUSTERS = 4
clustered = cluster_transactions(df, tx_feats, CLUSTER_FEATURE_COLS, n_clusters=N_CLUSTERS)
summary = summarise_clusters(clustered)
print("Cluster summary:")
summary

In [ ]:
fig_path = plot_clusters(clustered)
print(f"Saved: {fig_path}")

df_plot = clustered.copy()
df_plot["hour_float"] = df_plot["timestamp"].dt.hour + df_plot["timestamp"].dt.minute / 60
df_plot["log_amount"] = np.log1p(df_plot["amount"])
cmap = plt.get_cmap("tab10")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for cid in sorted(df_plot["cluster"].unique()):
    sub = df_plot[df_plot["cluster"]==cid]
    axes[0].scatter(sub["hour_float"], sub["log_amount"], alpha=0.4, s=15, color=cmap(cid), label=f"Cluster {cid}")
axes[0].set_xlabel("Time of day"); axes[0].set_ylabel("log(1+amount)"); axes[0].set_title("Payment Clusters"); axes[0].legend(); axes[0].grid(alpha=0.3)

cluster_dir = clustered.groupby(["cluster","direction"])["amount"].sum().unstack(fill_value=0).div(1e6)
cluster_dir.plot(kind="bar", ax=axes[1], color={"INFLOW": "#2ca02c", "OUTFLOW": "#d62728"}, edgecolor="white")
axes[1].set_title("Cluster Volume by Direction"); axes[1].set_ylabel("Total (M)"); axes[1].tick_params(axis="x", rotation=0); axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Connecting Anomalies to Payment Clusters

Identify which payment clusters are responsible for the detected anomaly buckets.

In [ ]:
anomaly_buckets = bucket_with_anomalies[bucket_with_anomalies["is_anomaly"]]["bucket"].values

clustered_copy = clustered.copy()
clustered_copy["bucket"] = clustered_copy["timestamp"].dt.floor("5min")
anomalous_txns = clustered_copy[clustered_copy["bucket"].isin(anomaly_buckets)]

print(f"Transactions in anomaly windows: {len(anomalous_txns)}")
print("
Cluster distribution during anomalies:")
print(anomalous_txns["cluster"].value_counts())
print("
Payment system distribution:")
print(anomalous_txns["payment_system"].value_counts())
print("
Direction distribution:")
print(anomalous_txns["direction"].value_counts())

## 7. Save Processed Data

In [ ]:
save_processed(liq, "../data/processed/liquidity_position.csv")
save_processed(bucket_with_anomalies, "../data/processed/anomaly_results.csv")
save_processed(clustered, "../data/processed/clustered_transactions.csv")
save_processed(summary, "../data/processed/cluster_summary.csv")
print("All processed datasets saved to data/processed/")

## Summary

| Component | Result |
|-----------|--------|
| Transactions loaded | 2,000 |
| 5-min time buckets | 132 |
| Anomalous buckets (Isolation Forest) | ~5% |
| Payment clusters (K-Means) | 4 |

**Key findings:**
- Corporate outflows dominate the afternoon stress window (13:30–14:30)
- Securities settlement payments (TARGET2/SECURITIES) drive morning and mid-afternoon liquidity spikes
- K-Means identifies distinct clusters: intraday corporate outflows, settlement inflows, interbank, and CCP activity
- Isolation Forest reliably flags unusual net-flow buckets during the simulated stress period